# 165 — Activation Subspace Analysis

Analyze the subspace structure of activations: PCA, subspace overlap across
layers, projection tracking, null space analysis, and component subspaces.

## Why This Matters

Activation subspace characterizes the patterns and statistics of internal model activations. Activation structure reveals how the model processes information — which components are active, how sparse the computation is, and what geometric patterns emerge.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_subspace_analysis import (
    activation_pca,
    subspace_overlap,
    projection_analysis,
    null_space_analysis,
    component_subspace_analysis,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
for l in range(4):
    result = activation_pca(model, tokens, layer=l, n_components=5)
    print(f"Layer {l}: eff_dim={result['effective_dimensionality']}  "
          f"top5_var={result['cumulative_variance'][-1]:.3f}  "
          f"var_explained={[f'{v:.3f}' for v in result['explained_variance']]}")

In [ ]:
result = subspace_overlap(model, tokens)
for p in result['per_pair']:
    print(f"L{p['layer_i']}<->L{p['layer_j']}: overlap={p['overlap']:.3f}  "
          f"max={p['max_overlap']:.3f}")

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = projection_analysis(model, tokens, direction)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: mean_proj={p['mean_projection']:+.4f}  "
          f"std={p['std_projection']:.4f}  range=[{p['min_projection']:.3f}, {p['max_projection']:.3f}]")

In [ ]:
for l in range(4):
    result = null_space_analysis(model, tokens, layer=l)
    print(f"Layer {l}: utilized={result['utilized_dims']}/{result['utilized_dims']+result['null_dims']}  "
          f"fraction={result['utilization_fraction']:.2f}  "
          f"sv_range=[{result['min_singular_value']:.4f}, {result['max_singular_value']:.4f}]")

In [ ]:
for l in range(4):
    result = component_subspace_analysis(model, tokens, layer=l)
    print(f"Layer {l}: attn_rank={result['attn_rank']:.1f}  mlp_rank={result['mlp_rank']:.1f}  "
          f"shared={result['shared_subspace']:.3f}  ortho={result['orthogonal_fraction']:.3f}")